# AI Workforce Statements ? Notebook 02: Transcript Inspection

This UC Berkeley project studies how public companies discuss AI adoption, workforce reductions, hiring, and productivity during earnings calls. The eventual statement dataset covers 2018?2025.

Only **Part 0 and Part 1** are implemented: local setup, metadata inspection, and a small transcript preview. Execute cells manually in VS Code; inspect and approve each Part before proceeding. Do not use Run All. No classification or causal attribution is performed. Keyword co-occurrence is not evidence of causation.

The professor's **RA Guide is the methodological authority**. It was not found in the project documentation during preparation. Consult it before later codebook and coding work; this notebook introduces no coding definitions.

The researcher-provided description of `Bose345/sp500_earnings_transcripts` reports approximately 33,362 calls from 2005?2025. This is an unverified reference expectation, not a finding. No network access, downloads, or installations are performed. The first records are a convenience preview, not a representative sample; no 2018?2025 filter is applied here.

## Part 0  Setup and Configuration


In [42]:
from pathlib import Path
import sys
import json
import reprlib

try:
    import pyarrow as pa
    import pyarrow.parquet as pq
except ImportError:
    pa = pq = None
    print("PyArrow unavailable: select the project .venv kernel and check dependencies. Nothing installed.")

In [43]:
PROJECT_ROOT_OVERRIDE = None  # Set to an absolute Path if VS Code starts outside the project.
TRANSCRIPT_RELATIVE_PATH = Path("data/raw/sp500_earnings_transcripts/part-0.parquet")
PREVIEW_RELATIVE_PATH = Path("data/preview/transcript_preview.txt")
PREVIEW_CALLS = 2
PREVIEW_SEGMENTS = 3
EXCERPT_CHARS = 600
MAX_EXPORT_BYTES = 50_000
EXPORT_PREVIEW = False
ALLOW_PREVIEW_OVERWRITE = False
EXPECTED_FIELDS = ["symbol", "company_name", "company_id", "year", "quarter",
                   "date", "content", "structured_content"]
REFERENCE_CALL_COUNT = 33_362  # Approximate handoff reference only.

# Configure only from observed keys in 1.2; no speaker-role inference.
SEGMENTS_PATH = ()  # Dict keys leading to a segment list, if wrapped.
SPEAKER_FIELD = None  # None uses only the explicit key "speaker".
TEXT_FIELD = None     # None uses only the explicit key "text".

if PROJECT_ROOT_OVERRIDE is not None:
    PROJECT_ROOT = Path(PROJECT_ROOT_OVERRIDE).expanduser().resolve()
else:
    current = Path.cwd().resolve()
    PROJECT_ROOT = next((p for p in [current, *current.parents]
                         if (p / "code/notebooks").is_dir()
                         and (p / "data").is_dir()
                         and (p / "README.md").is_file()), None)
if PROJECT_ROOT is None or not (PROJECT_ROOT / "code/notebooks").is_dir():
    raise RuntimeError("Project root not found. Set PROJECT_ROOT_OVERRIDE to the existing root.")
TRANSCRIPT_PATH = PROJECT_ROOT / TRANSCRIPT_RELATIVE_PATH
PREVIEW_PATH = PROJECT_ROOT / PREVIEW_RELATIVE_PATH
using_project_venv = Path(sys.prefix).resolve() == (PROJECT_ROOT / ".venv").resolve()
source_exists = TRANSCRIPT_PATH.is_file()
READY_FOR_PART_1 = pa is not None and using_project_venv and source_exists
print("Project root:", PROJECT_ROOT)
print("Python interpreter:", sys.executable)
print("Expected interpreter:", PROJECT_ROOT / ".venv/Scripts/python.exe")
print("Project .venv selected:", using_project_venv)
print("PyArrow version:", pa.__version__ if pa is not None else "unavailable")
print("Transcript path:", TRANSCRIPT_PATH)
print("Optional TXT preview:", PREVIEW_PATH)
if source_exists:
    size = TRANSCRIPT_PATH.stat().st_size
    print(f"Source exists: {size:,} bytes ({size / 1024**3:.3f} GiB)")
else:
    print("Source missing. Place the authorized file at the configured path or update it. No download attempted.")
print("Ready for Part 1 metadata inspection:", READY_FOR_PART_1)
print("Readability is not yet tested. Review Part 0 before proceeding.")

Project root: C:\Users\KPdesktop\Desktop\AI Workforce Statements
Python interpreter: c:\Users\KPdesktop\Desktop\AI Workforce Statements\.venv\Scripts\python.exe
Expected interpreter: C:\Users\KPdesktop\Desktop\AI Workforce Statements\.venv\Scripts\python.exe
Project .venv selected: True
PyArrow version: 26.0.0
Transcript path: C:\Users\KPdesktop\Desktop\AI Workforce Statements\data\raw\sp500_earnings_transcripts\part-0.parquet
Optional TXT preview: C:\Users\KPdesktop\Desktop\AI Workforce Statements\data\preview\transcript_preview.txt
Source exists: 1,824,805,388 bytes (1.699 GiB)
Ready for Part 1 metadata inspection: True
Readability is not yet tested. Review Part 0 before proceeding.


## Part 1 Dataset Inspection and Small Transcript Preview

### 1.1 Parquet Metadata Inspection

Run after reviewing Part 0. Read the footer and schema. Rows are documented as calls; uniqueness and date coverage are not established here. Report differences from the handoff. Arrow schema output includes nested child fields; string-valued structured content requires record inspection in 1.2.

In [44]:
parquet_schema = None
if not READY_FOR_PART_1:
    raise RuntimeError("Resolve readiness messages and rerun Part 0 first.")
with pq.ParquetFile(TRANSCRIPT_PATH) as source:
    parquet_schema = source.schema_arrow
    rows = source.metadata.num_rows
    print(f"Source rows / documented calls: {rows:,}")
    print(f"Approximate handoff reference: {REFERENCE_CALL_COUNT:,}; difference: {rows - REFERENCE_CALL_COUNT:+,}")
    print("Row groups:", source.metadata.num_row_groups)
    print(f"File size: {TRANSCRIPT_PATH.stat().st_size:,} bytes")
    print("\nColumn names and data types:")
    for field in parquet_schema:
        print(f"  {field.name}: {field.type} (nullable={field.nullable})")
    print("\nFull schema, including nested children:")
    print(parquet_schema)
print("Missing expected fields:", sorted(set(EXPECTED_FIELDS) - set(parquet_schema.names)) or "None")
print("Unexpected fields:", sorted(set(parquet_schema.names) - set(EXPECTED_FIELDS)) or "None")
if "structured_content" in parquet_schema.names:
    print("structured_content field:", parquet_schema.field("structured_content"))
else:
    print("No structured_content column; segmentation is unavailable from this field.")
print("The handoff did not specify exact data types. Inspect actual types above.")

Source rows / documented calls: 33,362
Approximate handoff reference: 33,362; difference: +0
Row groups: 2
File size: 1,824,805,388 bytes

Column names and data types:
  symbol: string (nullable=True)
  quarter: int64 (nullable=True)
  year: int64 (nullable=True)
  date: string (nullable=True)
  content: string (nullable=True)
  structured_content: list<element: struct<speaker: string, text: string>> (nullable=True)
  company_name: string (nullable=True)
  company_id: double (nullable=True)

Full schema, including nested children:
symbol: string
quarter: int64
year: int64
date: string
content: string
structured_content: list<element: struct<speaker: string, text: string>>
  child 0, element: struct<speaker: string, text: string>
      child 0, speaker: string
      child 1, text: string
company_name: string
company_id: double
-- schema metadata --
pandas: '{"index_columns": [], "column_indexes": [], "columns": [{"name":' + 1011
Missing expected fields: None
Unexpected fields: None
stru

### 1.2 Small In-Notebook Preview

Read only the first batch of at most **two calls**, selecting existing expected columns. No full-table or pandas conversion is used. PyArrow may decompress underlying pages; the returned batch is bounded, not the physical byte-level read.

Inspect the bounded structural diagnostic first. Valid JSON strings are decoded. Lists of dictionaries with explicit string `speaker` and `text` fields are displayed directly. For other keys or wrappers, use the observed representation to set `SEGMENTS_PATH`, `SPEAKER_FIELD`, and `TEXT_FIELD` in Part 0 and rerun configuration and Part 1. No alternative keys or boundaries are guessed. Unsupported representations require researcher review; a bounded unsegmented `content` fallback may be shown with an unidentified speaker.

Excerpts are original string slices, without wording normalization. `[excerpt ends]` is an added display label. Speaker roles and Prepared Remarks/Q&A labels are not inferred. Export uses only captured preview excerpts.

In [45]:
preview_records = []
preview_text = ""  # Prevent stale export after a failed preview.
if parquet_schema is None:
    raise RuntimeError("Run and inspect 1.1 first.")
if not (1 <= PREVIEW_CALLS <= 2 and 1 <= PREVIEW_SEGMENTS <= 5 and 1 <= EXCERPT_CHARS <= 1000):
    raise ValueError("Use 1?2 calls, 1?5 segments, and 1?1000 characters per excerpt.")
columns = [name for name in EXPECTED_FIELDS if name in parquet_schema.names]
if not columns:
    raise RuntimeError("No expected columns exist. Review the schema.")
with pq.ParquetFile(TRANSCRIPT_PATH) as source:
    batches = source.iter_batches(batch_size=PREVIEW_CALLS, columns=columns, use_threads=False)
    batch = next(batches, None)
    preview_records = batch.to_pylist() if batch is not None else []

bounded = reprlib.Repr()  # Bounded diagnostics only; excerpts remain original strings.
bounded.maxstring = bounded.maxother = 300
bounded.maxlist = bounded.maxtuple = 2
bounded.maxdict = 6
bounded.maxlevel = 3
preview_blocks = []
for row_index, record in enumerate(preview_records):
    header = "\n".join([
        f"Source: {TRANSCRIPT_PATH.name}; zero-based source row: {row_index}",
        f"Ticker: {bounded.repr(record.get('symbol'))}; company: {bounded.repr(record.get('company_name'))}",
        f"Date: {bounded.repr(record.get('date'))}",
        f"Year: {bounded.repr(record.get('year'))}; quarter: {bounded.repr(record.get('quarter'))}",
    ])
    print("\n" + "=" * 72 + "\n" + header)
    structured = record.get("structured_content")
    print("structured_content Python type:", type(structured).__name__)
    if isinstance(structured, str):
        try:
            structured = json.loads(structured)
            print("Decoded JSON type:", type(structured).__name__)
        except (json.JSONDecodeError, RecursionError):
            print("Not decodable JSON; no speaker boundaries inferred.")
    print("Bounded structure:", bounded.repr(structured))
    segments = structured
    for key in SEGMENTS_PATH:
        segments = segments.get(key) if isinstance(segments, dict) else None
    excerpts = []
    if isinstance(segments, list):
        print("Observed segment-list length:", len(segments))
        for segment_index, segment in enumerate(segments[:PREVIEW_SEGMENTS]):
            if not isinstance(segment, dict):
                print(f"Segment {segment_index}: not a dictionary; inspect representation.")
                continue
            print(f"Segment {segment_index} keys:", bounded.repr(list(segment)))
            speaker_key = SPEAKER_FIELD if SPEAKER_FIELD is not None else "speaker"
            text_key = TEXT_FIELD if TEXT_FIELD is not None else "text"
            speaker, original = segment.get(speaker_key), segment.get(text_key)
            if not isinstance(speaker, str) or not isinstance(original, str):
                print("No confirmed string speaker/text pair at configured keys. Review mapping.")
                continue
            excerpts.append(f"Segment {segment_index}; speaker (source field {speaker_key!r}): {speaker[:200]}\n"
                            + original[:EXCERPT_CHARS] + "\n[excerpt ends]")
    else:
        print("No segment list at configured path; review structure before mapping.")
    if not excerpts:
        content = record.get("content")
        if isinstance(content, str):
            excerpts.append("Speaker: unidentified (unsegmented content fallback)\n"
                            + content[:EXCERPT_CHARS] + "\n[excerpt ends]")
        else:
            print("No usable text excerpt; stop and review schema/representation.")
    preview_blocks.append(header + "\n\n" + ("\n\n".join(excerpts) or "No usable excerpt."))
    print("\n\n".join(excerpts))
preview_text = ("Human inspection only ? first source records, not an analysis dataset.\n"
                "Excerpts may be truncated; labels outside excerpts are added for display.\n\n"
                + "\n\n".join(preview_blocks)) if preview_records else ""
print(f"\nPreviewed calls: {len(preview_records)}. No files written.")
if not preview_records:
    print("No rows returned.")


Source: part-0.parquet; zero-based source row: 0
Ticker: 'A'; company: 'Agilent Technologies, Inc.'
Date: '2020-11-23 16:30:00'
Year: 2020; quarter: 4
structured_content Python type: list
Bounded structure: [{'speaker': 'Operator', 'text': "Good afternoon, and welcome to the Agilent Technologies Fourth Quarter Earnings Conference Call. All lines have been placed on mute to prevent any ...u. And now, I'd like to introduce you to the host for today's conference, Ankur Dhingra, Vice President of Investor Relations. Sir, please go ahead."}, {'speaker': 'Ankur Dhingra', 'text': "Thank you, and welcome everyone to Agilent's fourth quarter and full-year conference call for fiscal year 2020. With me are Mike McMullen, Agilent'...three group Presidents, followed by a Q&A. We look forward to having you join us on December 9. And now, I would like to turn the call over to Mike."}, ...]
Observed segment-list length: 113
Segment 0 keys: ['speaker', 'text']
Segment 1 keys: ['speaker', 'text']
Segme

### 1.3 Optional TXT Preview Export

Disabled by default. After inspecting 1.2, manually change `EXPORT_PREVIEW` to `True` **in the next cell** and run that cell. Set the separate overwrite flag there only if you explicitly approve replacing an existing preview. Permissions reset after an attempted export.

Export only captured headers and original excerpts from 1.2. No fresh source read occurs. UTF-8 output must be below 50 KB; reduce preview limits if needed. The directory is created only after authorization and checks pass. This TXT is for human inspection, not the final analysis dataset; nothing is saved to `data/clean/`.

In [46]:
EXPORT_PREVIEW = False  # Manually change to True to authorize this export.
ALLOW_PREVIEW_OVERWRITE = False  # Separate explicit approval to replace a file.

if not EXPORT_PREVIEW:
    print("TXT export disabled. No directory or file created.")
else:
    try:
        if not preview_records or not preview_text:
            raise RuntimeError("Run and review 1.2 successfully before exporting.")
        payload = preview_text.encode("utf-8")
        if len(payload) >= min(MAX_EXPORT_BYTES, 50_000):
            raise ValueError("Preview reaches byte limit. Reduce limits and rerun 1.2.")
        if PREVIEW_PATH.exists() and not ALLOW_PREVIEW_OVERWRITE:
            raise FileExistsError(f"Refusing overwrite: {PREVIEW_PATH}. Explicit approval required.")
        PREVIEW_PATH.parent.mkdir(parents=True, exist_ok=True)
        with PREVIEW_PATH.open("wb" if ALLOW_PREVIEW_OVERWRITE else "xb") as output:
            output.write(payload)
        print(f"Exported {len(payload):,} bytes to {PREVIEW_PATH}")
    finally:
        EXPORT_PREVIEW = False
        ALLOW_PREVIEW_OVERWRITE = False

TXT export disabled. No directory or file created.


### 1.4 Findings

- The Parquet loaded successfully: **33,362 calls, 8 fields, approximately 1.70 GiB** (1,824,805,388 bytes).
- The observed structured_content is a list of speaker-level segments with speaker names and original text.
- The inspected Agilent calls dated 2020-11-23 and 2020-08-18 contain **113 and 106 speaker segments**, respectively.
- Speaker roles and Prepared Remarks / Q&A are not explicitly structured as fields and require further verification.
- The dataset is ready for keyword-based statement discovery. This establishes inspection readiness, not validated classifications or coverage.

These findings use existing executed Part 1 outputs; Part 1 was not rerun.


## Part 2 - Reproducible Validation Sampling

Implement supplied sampling protocol: target 100 AI + Workforce proximity candidates, 100 Workforce-only candidates, and 100 AI-only candidates. These are preliminary sampling groups, not verified statement classes or evidence of causation. No human label is assigned automatically.

Run 2.1, 2.2, and 2.3 manually in order. Part 0 supplies the existing root, path, and PyArrow imports; if the kernel restarted, run Part 0 first. Parts 0-1 need no changes. This sample is reserved for human-coded validation: do not reuse it for codebook development or prompt tuning. Use separate pilot examples.

Operational retrieval rules (not new coding definitions):
- AI + Workforce proximity: call date in 2023-2025; an AI keyword and workforce keyword separated by at most 50 intervening whitespace-delimited words in the same speaker segment.
- Workforce-only: call date in 2018-2025; a workforce-matching segment from a call with no AI keyword anywhere in either available full content or speaker text. Missing/incomplete text without a usable full-content fallback cannot establish absence and is excluded.
- AI-only: call date in 2023-2025; an AI-matching speaker segment with no workforce keyword anywhere in that segment. This applies the no-workforce condition to the candidate segment, not the entire call.
- Segments containing both types more than 50 words apart do not enter any group. A segment enters at most one group; final selection also allows at most one candidate per call across all groups.

A passage is an exact contextual slice within a source speaker segment, not an automatically adjudicated statement. No proximity is inferred across speaker boundaries. All keyword matches are lexical; semantic absence, speaker roles, and causal attribution require human verification.

### 2.1 - Sampling and Keyword Configuration

The existing AI/workforce dictionaries remain visible below. Regex matching is case-insensitive with word boundaries and whitespace/hyphen phrase separators. Automation need not involve AI; attrition and restructuring may concern other subjects. Acronyms, synonyms, negation, and non-keyword wording can produce false positives or misses.

Use a fixed seed and a two-stage probability-based selection with coverage constraints:
1. Scan every row group. Uniformly select one eligible segment per call per group using reservoir sampling; retain only compact source references, not transcript text.
2. Shuffle group/year strata each round. Within each stratum, favor companies with the fewest selections so far, randomize ties, then select a random eligible call. If a source industry/sector field exists, favor less-represented values among tied companies. Targets are 100 per group, at most 1 per call and 3 per company across the whole sample. This seeks broad coverage; it is not a simple random sample of statements and does not support unweighted prevalence estimates.

All eligible calls can reach selection; there is no first-firm or row-group cutoff. Call IDs conservatively combine source company identity and calendar date, so same-company same-day duplicates share a cap. Company identity uses company_id, then ticker, then name; missing identities are excluded and reported. These are source identifiers, not verified historical crosswalk matches.

Known pilot exclusions: Part 1 source rows 0 and 1 and, if present, calls listed in the old transcript_candidates.csv. Exclude entire corresponding calls. Additional pilot source rows can be added below before sampling. If other pilot material exists, supply its identifiers before treating the sample as held out. No pilot file is modified. Source-version mismatches in the old CSV stop the scan for review.

No equal industry quotas are invented. The observed schema has no industry field; optional exact source fields industry, sector, or gics_sector are used only if present. Missing industries remain missing. Failure to meet 100 is reported; caps and eligibility are never silently relaxed.

In [47]:
import re
import csv
import io
import math
import random
from bisect import bisect_right
from datetime import date, datetime
from collections import Counter, defaultdict

SAMPLING_SEED = 20261024
TARGET_PER_GROUP = 100
MAX_PER_COMPANY = 3  # Across all three groups; one per call is enforced separately.
DISCOVERY_BATCH_SIZE = 16
STUDY_START, STUDY_END = date(2018, 1, 1), date(2025, 12, 31)
AI_PERIOD_START = date(2023, 1, 1)
MAX_WORD_GAP = 50
CONTEXT_WORDS = 40  # On each side of the anchor/pair, inside its speaker segment.
PREVIEW_PER_GROUP = 1
CANDIDATE_CSV_PATH = PROJECT_ROOT / "data/clean/transcript_candidates_300.csv"
PILOT_CSV_PATH = PROJECT_ROOT / "data/clean/transcript_candidates.csv"
PILOT_SOURCE_ROWS = {0, 1}  # Known Part 1 examples; extend for other pilot calls.
EXPORT_CANDIDATES = False
MAX_CSV_BYTES = 5_000_000
GROUPS = ("AI + Workforce proximity", "Workforce-only", "AI-only")

AI_KEYWORDS = {
    "AI": r"ai",
    "artificial intelligence": r"artificial[\s-]+intelligence",
    "automation": r"automat(?:ion|ions|e|es|ed|ing)",
    "machine learning": r"machine[\s-]+learning",
    "generative AI": r"generative[\s-]+(?:ai|artificial[\s-]+intelligence)|gen[\s-]?ai",
    "large language model": r"large[\s-]+language[\s-]+models?|llms?",
}
WORKFORCE_KEYWORDS = {
    "layoff": r"lay[\s-]?offs?|laid[\s-]+off",
    "headcount": r"head[\s-]?counts?",
    "hiring": r"hir(?:e|es|ed|ing)",
    "hiring freeze": r"hiring[\s-]+freez(?:e|es)|freez(?:e|es|ing)[\s-]+hiring",
    "restructuring": r"restructur(?:e|es|ed|ing|ings)",
    "attrition": r"attrition",
    "workforce": r"work[\s-]?forces?",
    "job cuts": r"job[\s-]+cuts?|cut(?:ting)?[\s-]+jobs",
    "workforce reduction": r"(?:work[\s-]?force|staff|head[\s-]?count)[\s-]+reductions?",
    "redundancy": r"redundan(?:cy|cies|t)",
    "downsizing": r"downsiz(?:e|es|ed|ing)",
}
AI_PATTERNS = {label: re.compile(r"\b(?:" + pattern + r")\b", re.IGNORECASE)
               for label, pattern in AI_KEYWORDS.items()}
WORKFORCE_PATTERNS = {label: re.compile(r"\b(?:" + pattern + r")\b", re.IGNORECASE)
                      for label, pattern in WORKFORCE_KEYWORDS.items()}

print("Seed:", SAMPLING_SEED)
print("Targets:", {group: TARGET_PER_GROUP for group in GROUPS})
print("Caps: one per call; at most", MAX_PER_COMPANY, "per company across all groups")
print("Validation CSV:", CANDIDATE_CSV_PATH)
for name, dictionary in [("AI", AI_KEYWORDS), ("Workforce", WORKFORCE_KEYWORDS)]:
    print(name, "keywords:", dictionary)
print("No source scan or export performed by this cell.")

Seed: 20261024
Targets: {'AI + Workforce proximity': 100, 'Workforce-only': 100, 'AI-only': 100}
Caps: one per call; at most 3 per company across all groups
Validation CSV: C:\Users\KPdesktop\Desktop\AI Workforce Statements\data\clean\transcript_candidates_300.csv
AI keywords: {'AI': 'ai', 'artificial intelligence': 'artificial[\\s-]+intelligence', 'automation': 'automat(?:ion|ions|e|es|ed|ing)', 'machine learning': 'machine[\\s-]+learning', 'generative AI': 'generative[\\s-]+(?:ai|artificial[\\s-]+intelligence)|gen[\\s-]?ai', 'large language model': 'large[\\s-]+language[\\s-]+models?|llms?'}
Workforce keywords: {'layoff': 'lay[\\s-]?offs?|laid[\\s-]+off', 'headcount': 'head[\\s-]?counts?', 'hiring': 'hir(?:e|es|ed|ing)', 'hiring freeze': 'hiring[\\s-]+freez(?:e|es)|freez(?:e|es|ing)[\\s-]+hiring', 'restructuring': 'restructur(?:e|es|ed|ing|ings)', 'attrition': 'attrition', 'workforce': 'work[\\s-]?forces?', 'job cuts': 'job[\\s-]+cuts?|cut(?:ting)?[\\s-]+jobs', 'workforce reduction':

### 2.2 - Full Scan and Reproducible Selection

The first pass projects metadata, structured_content, and content in small batches across **all row groups**. Full content is needed to check call-level AI absence for Workforce-only. Only one batch and compact call/group references are retained; no full dataset is loaded into pandas or concatenated. A second incremental pass retrieves only the selected passages into the final in-memory sample.

Dates are parsed from the observed ISO call-date strings, without a fiscal-year fallback. Invalid dates, source structures, missing identity, and pilot exclusions are counted. A source size/mtime check prevents exporting against a changed file.

Word distance is the number of whitespace-delimited tokens strictly between the two keyword spans; overlapping or adjacent spans have distance zero. All occurrences are checked. The closest qualifying pair (earliest on ties) anchors a proximity passage; a first keyword anchors other groups. Forty tokens of context are kept on either side when available. Original character offsets are zero-based, end-exclusive. No rewriting or truncation through the anchor is performed.

Each call/group reservoir samples its eligible segments uniformly. The final round-robin group/year selection randomizes company and call choices subject to coverage priorities and caps. Same seed, source, keyword rules, exclusions, and Python environment produce the same sample. Coverage constraints can yield shortfalls even when raw segment counts exceed 100; diagnostics distinguish those cases.

In [48]:
sampling_complete = False
candidate_sample = []
rng_segments = random.Random(SAMPLING_SEED)
rng_select = random.Random(SAMPLING_SEED + 1)
eligible_segments = Counter({g: 0 for g in GROUPS})
scan_counts = Counter()
call_candidates = []  # Compact references only; at most one per source call/group.
if pa is None or pq is None or not TRANSCRIPT_PATH.is_file():
    raise RuntimeError("Run Part 0 setup and resolve source readiness first.")
if not (1 <= DISCOVERY_BATCH_SIZE <= 64 and TARGET_PER_GROUP == 100
        and 1 <= MAX_PER_COMPANY <= 3 and 0 <= CONTEXT_WORDS <= 100):
    raise ValueError("Keep the 100/group target, batch <=64, company cap <=3, context <=100.")
source_stat = TRANSCRIPT_PATH.stat()
source_signature = (source_stat.st_size, source_stat.st_mtime_ns)
source_identifier = str(TRANSCRIPT_PATH.relative_to(PROJECT_ROOT))
pilot_rows = set(PILOT_SOURCE_ROWS)
pilot_file_present = PILOT_CSV_PATH.is_file()
if pilot_file_present:
    with PILOT_CSV_PATH.open(encoding="utf-8-sig", newline="") as pilot_file:
        for pilot in csv.DictReader(pilot_file):
            if pilot.get("source_file", "").replace("\\", "/") != source_identifier.replace("\\", "/"):
                raise ValueError("Pilot CSV source differs; review its identifiers before sampling.")
            if (int(pilot["source_size_bytes"]), int(pilot["source_mtime_ns"])) != source_signature:
                raise ValueError("Pilot CSV source version differs; reconcile exclusions before sampling.")
            pilot_rows.add(int(pilot["source_row"]))

def keyword_hits(text, patterns):
    """All occurrences, including repeated terms needed for proximity."""
    return sorted([(m.start(), m.end(), label)
                   for label, pattern in patterns.items() for m in pattern.finditer(text)])

def company_identity(record):
    """Use existing identifiers only, with explicit fallbacks."""
    value = record.get("company_id")
    if isinstance(value, (int, float)) and math.isfinite(value):
        return "company_id:" + str(value)
    for field in ("symbol", "company_name"):
        value = record.get(field)
        if isinstance(value, str) and value.strip():
            return field + ":" + value.strip()
    return None

def passage_anchor(text, ai_hits, workforce_hits):
    """Closest pair within 50 intervening whitespace tokens, else no anchor."""
    word_starts = [m.start() for m in re.finditer(r"\S+", text)]
    best = None
    # Keywords per speaker segment are usually sparse; retain no pair matrix.
    for a_start, a_end, _ in ai_hits:
        a_first = bisect_right(word_starts, a_start) - 1
        a_last = bisect_right(word_starts, a_end - 1) - 1
        for w_start, w_end, _ in workforce_hits:
            w_first = bisect_right(word_starts, w_start) - 1
            w_last = bisect_right(word_starts, w_end - 1) - 1
            gap = max(0, w_first - a_last - 1, a_first - w_last - 1)
            pair = (gap, min(a_start, w_start), max(a_end, w_end))
            if gap <= MAX_WORD_GAP and (best is None or pair < best):
                best = pair
    return best

with pq.ParquetFile(TRANSCRIPT_PATH) as source:
    required = ["symbol", "company_name", "company_id", "date", "year", "quarter",
                "structured_content", "content"]
    missing = sorted(set(required) - set(source.schema_arrow.names))
    if missing:
        raise ValueError(f"Required fields missing: {missing}")
    industry_field = next((name for name in ("industry", "sector", "gics_sector")
                           if name in source.schema_arrow.names), None)
    columns = required + ([industry_field] if industry_field else [])
    total_source_rows = source.metadata.num_rows
    if any(not isinstance(row, int) or not 0 <= row < total_source_rows for row in pilot_rows):
        raise ValueError("Pilot source rows must be valid zero-based rows in this source.")
    # Metadata-only pass resolves pilot rows to call identities, also excluding duplicate calls.
    excluded_call_keys = set()
    row_number = 0
    for batch in source.iter_batches(batch_size=DISCOVERY_BATCH_SIZE,
                                     columns=["company_id", "symbol", "company_name", "date"],
                                     use_threads=False):
        for record in batch.to_pylist():
            if row_number in pilot_rows:
                try:
                    pilot_date = datetime.fromisoformat(record["date"]).date()
                except (TypeError, ValueError):
                    raise ValueError("Cannot resolve a pilot call date; review exclusions.")
                company = company_identity(record)
                if company is None:
                    raise ValueError("Cannot resolve a pilot company; review exclusions.")
                excluded_call_keys.add(company + "|" + pilot_date.isoformat())
            row_number += 1

    for row_group in range(source.metadata.num_row_groups):
        row_in_group = -1
        for batch in source.iter_batches(batch_size=DISCOVERY_BATCH_SIZE,
                                         row_groups=[row_group], columns=columns, use_threads=False):
            for record in batch.to_pylist():
                source_row = scan_counts["source_rows_scanned"]
                scan_counts["source_rows_scanned"] += 1
                row_in_group += 1
                try:
                    call_date = datetime.fromisoformat(record["date"]).date()
                except (TypeError, ValueError):
                    scan_counts["invalid_dates"] += 1
                    continue
                if not STUDY_START <= call_date <= STUDY_END:
                    scan_counts["outside_2018_2025"] += 1
                    continue
                company = company_identity(record)
                if company is None:
                    scan_counts["missing_company_identity"] += 1
                    continue
                call_key = company + "|" + call_date.isoformat()
                if call_key in excluded_call_keys:
                    scan_counts["pilot_call_rows_excluded"] += 1
                    continue
                scan_counts["study_call_rows_after_exclusions"] += 1
                segments = record["structured_content"]
                if not isinstance(segments, list) or not segments:
                    scan_counts["missing_or_unsupported_segments"] += 1
                    continue
                full_content = record["content"]
                content_available = isinstance(full_content, str) and bool(full_content.strip())
                call_has_ai = content_available and any(p.search(full_content) for p in AI_PATTERNS.values())
                complete_segments = True
                local_seen = Counter()
                local_choices = {}
                for segment_index, segment in enumerate(segments):
                    text = segment.get("text") if isinstance(segment, dict) else None
                    if not isinstance(text, str):
                        complete_segments = False
                        scan_counts["invalid_segment_text"] += 1
                        continue
                    if not text.strip():
                        scan_counts["empty_segment_text"] += 1
                        continue
                    ai_hits = keyword_hits(text, AI_PATTERNS)
                    wf_hits = keyword_hits(text, WORKFORCE_PATTERNS)
                    call_has_ai = call_has_ai or bool(ai_hits)
                    group = None
                    if ai_hits and wf_hits and call_date >= AI_PERIOD_START:
                        pair = passage_anchor(text, ai_hits, wf_hits)
                        if pair is not None:
                            gap, anchor_start, anchor_end = pair
                            group = GROUPS[0]
                        else:
                            scan_counts["both_terms_beyond_50_words"] += 1
                    elif wf_hits and not ai_hits:
                        group, gap = GROUPS[1], None  # Tentative until entire call is checked.
                        anchor_start, anchor_end, _ = wf_hits[0]
                    elif ai_hits and not wf_hits and call_date >= AI_PERIOD_START:
                        group, gap = GROUPS[2], None
                        anchor_start, anchor_end, _ = ai_hits[0]
                    if group is None:
                        continue
                    local_seen[group] += 1
                    if rng_segments.randrange(local_seen[group]) == 0:
                        industry = record.get(industry_field) if industry_field else None
                        local_choices[group] = {
                            "sampling_group": group, "source_row": source_row,
                            "row_group": row_group, "row_in_group": row_in_group,
                            "segment_index": segment_index, "company_key": company,
                            "call_key": call_key, "call_year": call_date.year,
                            "industry": str(industry) if industry is not None else "",
                            "anchor_start": anchor_start, "anchor_end": anchor_end,
                            "ai_workforce_word_gap": gap,
                        }
                # Require absence over all available call text, not just candidate segment.
                if call_has_ai or not (content_available or complete_segments):
                    if GROUPS[1] in local_seen:
                        scan_counts["tentative_workforce_segments_rejected"] += local_seen[GROUPS[1]]
                    local_seen.pop(GROUPS[1], None)
                    local_choices.pop(GROUPS[1], None)
                for group, count in local_seen.items():
                    eligible_segments[group] += count
                    call_candidates.append(local_choices[group])
        print(f"Scanned row group {row_group + 1}/{source.metadata.num_row_groups}")
    if scan_counts["source_rows_scanned"] != total_source_rows:
        raise RuntimeError("Full source scan did not match metadata row count.")

# Compact candidate references span all eligible calls; no first-firm cutoff.
pools = defaultdict(list)
for candidate in call_candidates:
    pools[(candidate["sampling_group"], candidate["call_year"])].append(candidate)
selected_refs = []
selected_counts = Counter()
company_counts = Counter()
industry_counts = Counter()
used_calls = set()
strata = sorted(pools)
while True:
    made_selection = False
    round_strata = strata.copy()
    rng_select.shuffle(round_strata)
    for group, year in round_strata:
        if selected_counts[group] >= TARGET_PER_GROUP:
            continue
        available = [c for c in pools[(group, year)]
                     if c["call_key"] not in used_calls and company_counts[c["company_key"]] < MAX_PER_COMPANY]
        if not available:
            continue
        by_company = defaultdict(list)
        for candidate in available:
            by_company[candidate["company_key"]].append(candidate)
        least_used = min(company_counts[company] for company in by_company)
        company_options = sorted(company for company in by_company if company_counts[company] == least_used)
        if industry_field:
            industry_score = {company: min(industry_counts[c["industry"]] for c in by_company[company])
                              for company in company_options}
            least_industry = min(industry_score.values())
            company_options = [company for company in company_options if industry_score[company] == least_industry]
        company = rng_select.choice(company_options)
        # Duplicate source rows with the same company/date count as one call choice.
        by_call = defaultdict(list)
        for candidate in by_company[company]:
            by_call[candidate["call_key"]].append(candidate)
        call_key = rng_select.choice(sorted(by_call))
        chosen = rng_select.choice(by_call[call_key])
        selected_refs.append(chosen)
        selected_counts[group] += 1
        company_counts[company] += 1
        industry_counts[chosen["industry"]] += 1
        used_calls.add(call_key)
        made_selection = True
    if not made_selection:
        break

# Retrieve exact contextual passages only for the selected references.
selected_by_row = {c["source_row"]: c for c in selected_refs}
retrieved = {}
with pq.ParquetFile(TRANSCRIPT_PATH) as source:
    source_row = 0
    for batch in source.iter_batches(batch_size=DISCOVERY_BATCH_SIZE,
                                     columns=[name for name in columns if name != "content"],
                                     use_threads=False):
        for record in batch.to_pylist():
            selected = selected_by_row.get(source_row)
            source_row += 1
            if selected is None:
                continue
            segment = record["structured_content"][selected["segment_index"]]
            text = segment["text"]
            tokens = list(re.finditer(r"\S+", text))
            starts = [token.start() for token in tokens]
            left = max(0, bisect_right(starts, selected["anchor_start"]) - 1 - CONTEXT_WORDS)
            right = min(len(tokens) - 1, bisect_right(starts, selected["anchor_end"] - 1) - 1 + CONTEXT_WORDS)
            passage_start, passage_end = tokens[left].start(), tokens[right].end()
            passage = text[passage_start:passage_end]
            speaker = segment.get("speaker")
            result = dict(selected)
            result.update({
                "sample_id": f"validation-{SAMPLING_SEED}-{selected['source_row']}-{selected['segment_index']}",
                "sampling_seed": SAMPLING_SEED, "source_file": source_identifier,
                "source_size_bytes": source_signature[0], "source_mtime_ns": source_signature[1],
                "symbol": record["symbol"], "company_name": record["company_name"],
                "company_id": record["company_id"], "date": record["date"],
                "fiscal_year": record["year"], "quarter": record["quarter"],
                "speaker": speaker if isinstance(speaker, str) else "",
                "industry_source_field": industry_field or "",
                "matched_ai_keywords": json.dumps(sorted({h[2] for h in keyword_hits(passage, AI_PATTERNS)})),
                "matched_workforce_keywords": json.dumps(sorted({h[2] for h in keyword_hits(passage, WORKFORCE_PATTERNS)})),
                "passage_start": passage_start, "passage_end": passage_end,
                "segment_char_count": len(text), "original_passage": passage,
                "statement_class": "", "action": "", "cause_primary": "",
            })
            retrieved[selected["source_row"]] = result
candidate_sample = [retrieved[c["source_row"]] for c in selected_refs]
final_stat = TRANSCRIPT_PATH.stat()
if (final_stat.st_size, final_stat.st_mtime_ns) != source_signature:
    raise RuntimeError("Source changed during sampling; do not use these results.")
if len(candidate_sample) != len(selected_refs) or len(used_calls) != len(candidate_sample):
    raise RuntimeError("Selected references or call uniqueness failed verification.")
sampling_complete = True
print(f"Sampling complete: {len(candidate_sample)}/300 candidates. No files written.")

Scanned row group 1/2
Scanned row group 2/2
Sampling complete: 300/300 candidates. No files written.


### 2.3 - Coverage, Constraints, and Limited Preview

Counts distinguish eligible source segments, distinct eligible calls, and selected candidates after exclusions and caps. Year distributions use call date. Company identity fallbacks and same-day call identity are source-based approximations, not inferred corporate crosswalks. The report lists sampling constraints and every group shortfall; fewer than 100 eligible segments is distinguished from shortages of distinct calls/companies or competition between groups.

The limited display is for source/format verification only. Keep these candidates reserved for validation; do not use their content or eventual human labels to tune the codebook or extraction prompts. Industry values are shown only if present in the source.

In [50]:
if not sampling_complete:
    raise RuntimeError("Complete 2.2 successfully before viewing this sample.")
if not 0 <= PREVIEW_PER_GROUP <= 2:
    raise ValueError("Use 0-2 preview examples per group.")
print("Scan counters:", dict(sorted(scan_counts.items())))
print("Seed:", SAMPLING_SEED, "| cap per call: 1 | cap per company:", MAX_PER_COMPANY)
print("Pilot CSV found:", pilot_file_present, "| known pilot calls excluded:", len(excluded_call_keys))
print("Industry source field:", industry_field or "Unavailable; no industry assignments inferred")
for group in GROUPS:
    eligible = [c for c in call_candidates if c["sampling_group"] == group]
    chosen = [c for c in candidate_sample if c["sampling_group"] == group]
    shortfall = TARGET_PER_GROUP - len(chosen)
    print(f"\n{group}: selected {len(chosen)}/100; shortfall {shortfall}")
    print("Eligible segments:", eligible_segments[group],
          "| eligible distinct calls:", len({c["call_key"] for c in eligible}),
          "| eligible companies:", len({c["company_key"] for c in eligible}))
    print("Selected unique companies:", len({c["company_key"] for c in chosen}),
          "| unique calls:", len({c["call_key"] for c in chosen}))
    print("Eligible call years:", dict(sorted(Counter(year for _, year in
          {(c["call_key"], c["call_year"]) for c in eligible}).items())))
    print("Selected call years:", dict(sorted(Counter(c["call_year"] for c in chosen).items())))
    if shortfall:
        reason = ("Fewer than 100 eligible segments after retrieval rules and pilot exclusions."
                  if eligible_segments[group] < TARGET_PER_GROUP else
                  "Distinct-call/company capacity and shared caps limit this draw; no replacements invented.")
        print("SHORTFALL:", reason)
    for c in chosen[:PREVIEW_PER_GROUP]:
        print(f"\n{c['sample_id']} | {c['symbol']} | {c['date']} | {c['speaker'] or '[not provided]'}")
        print("AI keywords:", c["matched_ai_keywords"], "| Workforce keywords:", c["matched_workforce_keywords"])
        print("Word gap:", c["ai_workforce_word_gap"], "| source row/segment:",
              c["source_row"], c["segment_index"], "| character range:", c["passage_start"], c["passage_end"])
        print(c["original_passage"])
        print("[End of exact passage; context outside this range is omitted.]")
print("\nTOTAL:", len(candidate_sample), "| unique companies:", len(company_counts), "| unique calls:", len(used_calls))
print("Overall years:", dict(sorted(Counter(c["call_year"] for c in candidate_sample).items())))
print("Company identity sources:", dict(Counter(c["company_key"].split(":", 1)[0] for c in candidate_sample)))
print("Max selected per company:", max(company_counts.values(), default=0))
if industry_field:
    print("Source industry distribution:", dict(sorted(industry_counts.items())))
print("Constraints: keyword eligibility; date windows; known pilot exclusions; 1/call; 3/company; year/company coverage.")
print("This is a stratified, constrained validation sample, not an unbiased prevalence estimate.")
print("Human label fields remain empty.")

Scan counters: {'both_terms_beyond_50_words': 611, 'empty_segment_text': 1230, 'missing_or_unsupported_segments': 2, 'outside_2018_2025': 17857, 'pilot_call_rows_excluded': 2, 'source_rows_scanned': 33362, 'study_call_rows_after_exclusions': 15503, 'tentative_workforce_segments_rejected': 9348}
Seed: 20261024 | cap per call: 1 | cap per company: 3
Pilot CSV found: False | known pilot calls excluded: 2
Industry source field: Unavailable; no industry assignments inferred

AI + Workforce proximity: selected 100/100; shortfall 0
Eligible segments: 463 | eligible distinct calls: 354 | eligible companies: 150
Selected unique companies: 97 | unique calls: 100
Eligible call years: {2023: 126, 2024: 160, 2025: 68}
Selected call years: {2023: 34, 2024: 33, 2025: 33}

validation-20261024-7423-80 | PAYX | 2023-12-21 09:30:00 | John Gibson
AI keywords: ["AI"] | Workforce keywords: ["workforce"]
Word gap: 33 | source row/segment: 7423 80 | character range: 577 1201
to the point you may be making is 

### 2.4 CSV Export

The only output is data/clean/transcript_candidates_300.csv. Set EXPORT_CANDIDATES = True in the next cell only after reviewing the sample and any shortfalls. An incomplete sample may be exported as the available sample, with its size explicitly reported; no missing candidates are fabricated.

Export is disabled by default and **never overwrites an existing file**. The old transcript_candidates.csv is read only for known pilot exclusions and remains unchanged. Output includes sampling_group, keywords, original_passage, traceable source identifiers, seed, offsets, and empty statement_class/action/cause_primary fields. Preserve this sample as validation material, separate from pilot/tuning examples. Full transcripts are not exported.

In [ ]:
EXPORT_CANDIDATES = False  # Manually set True after reviewing counts and shortfalls.

if not EXPORT_CANDIDATES:
    print("Validation CSV export disabled. No files or directories created.")
else:
    try:
        if not sampling_complete or not candidate_sample:
            raise RuntimeError("Complete and review 2.2-2.3 first.")
        if len(candidate_sample) > 300:
            raise ValueError("Validation sample exceeds 300 candidates.")
        if CANDIDATE_CSV_PATH.exists():
            raise FileExistsError(f"Refusing to overwrite existing file: {CANDIDATE_CSV_PATH}")
        current_stat = TRANSCRIPT_PATH.stat()
        if (current_stat.st_size, current_stat.st_mtime_ns) != source_signature:
            raise RuntimeError("Source changed since sampling; review before export.")
        if any(c[label] != "" for c in candidate_sample
               for label in ("statement_class", "action", "cause_primary")):
            raise ValueError("Human label fields must be empty on initial export.")
        buffer = io.StringIO(newline="")
        writer = csv.DictWriter(buffer, fieldnames=list(candidate_sample[0]))
        writer.writeheader()
        writer.writerows(candidate_sample)
        payload = buffer.getvalue().encode("utf-8")
        if len(payload) > MAX_CSV_BYTES:
            raise ValueError("Sample CSV exceeds byte limit; review passage lengths before export.")
        CANDIDATE_CSV_PATH.parent.mkdir(parents=True, exist_ok=True)
        with CANDIDATE_CSV_PATH.open("xb") as output:
            output.write(payload)
        print(f"Exported {len(candidate_sample)}/300 candidates to {CANDIDATE_CSV_PATH}")
        print("Group counts:", dict(selected_counts))
        print("Shortfalls:", {g: TARGET_PER_GROUP - selected_counts[g] for g in GROUPS})
    finally:
        EXPORT_CANDIDATES = False

Exported 300/300 candidates to C:\Users\KPdesktop\Desktop\AI Workforce Statements\data\clean\transcript_candidates_300.csv
Group counts: {'AI-only': 100, 'AI + Workforce proximity': 100, 'Workforce-only': 100}
Shortfalls: {'AI + Workforce proximity': 0, 'Workforce-only': 0, 'AI-only': 0}


### 2.5 - Findings

**Protocol.** Target 100 AI + Workforce proximity candidates from 2023-2025 (within 50 intervening words in one speaker segment), 100 Workforce-only candidates from 2018-2025 calls with no detected AI mention, and 100 AI-only candidates from 2023-2025 segments without workforce content. These remain preliminary lexical sampling groups.

**Verified current file.** A read-only metadata check of data/clean/transcript_candidates_300.csv confirms **300 rows**, consistent with the stored execution outputs; the count was checked rather than inferred from its filename. There are 300 distinct sample IDs, 300 source rows/calls, and 296 distinct source company keys. No sampling cells were rerun.

| Sampling group | Current rows | Source company keys | Calls | Call-year distribution |
|---|---:|---:|---:|---|
| AI + Workforce proximity | 100 | 97 | 100 | 2023: 34; 2024: 33; 2025: 33 |
| Workforce-only | 100 | 100 | 100 | 2018: 13; 2019: 12; 2020: 12; 2021: 13; 2022: 12; 2023: 13; 2024: 13; 2025: 12 |
| AI-only | 100 | 100 | 100 | 2023: 34; 2024: 33; 2025: 33 |

Overall years: 2018: 13; 2019: 12; 2020: 12; 2021: 13; 2022: 12; 2023: 81; 2024: 79; 2025: 78. The executed scan covered both row groups and all 33,362 source rows. Seed: 20261024. The configured caps were one per call and three per company; the observed maximum is two per source company key. All groups meet the 100 target.

**Metadata.** Company ID and company name are each missing for 28 rows. Identity keys use company_id for 272 rows and ticker for 28. Industry and industry-source fields are missing for all 300. Ticker, call date, fiscal year, quarter, speaker name, and recorded source identifiers are populated. Speaker role and call section are not provided as structured CSV fields. All 300 statement_class/action/cause_primary entries remain blank by design.

**Pilot separation.** Stored outputs show two known Pilot calls excluded. The current CSV contains neither Part 1 source row 0 or 1 nor their Agilent ticker/date combinations (2020-11-23 and 2020-08-18): the known overlap has been addressed. The old transcript_candidates.csv is currently absent, matching the executed 'Pilot CSV found: False' message. This verifies the known exclusions, not every possible external Pilot artifact; any future Pilot input must pass the Part 3 non-overlap check. Validation passages were not used to develop this codebook.

**Limits.** Keyword candidates are not verified statements or evidence of AI causation. Year/company coverage priorities and caps mean this is not a simple random sample for prevalence estimates. Source company keys are not yet verified historical firms; ticker fallbacks may not resolve corporate identity. Company/industry metadata can be enriched through a later approved historical crosswalk. No merge or label assignment was performed.


## Part 3 - Codebook Development

### 3.1 - Coding Principles

The goal is a reproducible human protocol for management statements about the firm's own AI activity and workforce changes. 

- Include workforce reductions/layoffs, freezes, slower hiring, no-backfill, offshoring, headcount-affecting restructuring, and workforce growth. Include the firm's own AI investment, adoption, deployment, productivity effects, operational applications, and specific future plans. Both requires substantive content in both areas.
- Exclude generic employee praise, cost savings without workforce implications, vague AI enthusiasm, and customer-facing AI products without internal operational relevance.
- Require evidence of management status, role, and call section; analyst questions can supply context but are not the management statement. Missing evidence stays unresolved.
- Code only explicitly attributed causes. AI/layoff proximity, automation, efficiency, or time saved do not establish AI-caused workforce reductions.
- Preserve exact quotes, source/call/segment identifiers, and character offsets. Split multiple distinct actions into separate traceable records, without duplicating a single action for each keyword.
- Leave unsupported/inapplicable values blank and explain uncertainty in human notes. Pure AI leaves workforce action and cause fields blank.
- Keep validation passages out of codebook development and prompt tuning. No automatic labels are assigned.

### 3.2 - Codebook Draft

Review [docs/codebook.md](../../docs/codebook.md). It contains all required shared and AI fields, their definitions, allowed values, evidence rules, and two unfilled example slots per field with places for coding explanations.

No genuine example has been filled in yet. Complete examples only after reviewing independent Pilot passages. Confidence anchors, unranked causes, and other unspecified decisions remain explicit questions. No CSV labels or crosswalk links are changed by this section.

### 3.3 Example Review: Independent Pilot Only

No separate Pilot CSV was present when this section was prepared. The default path below therefore produces a clear placeholder. Do not point it at the reserved validation CSV or copy validation passages into a Pilot file.

When a separate Pilot file is available, set PILOT_REVIEW_PATH. Accepted columns are the prior candidate format (excerpt/excerpt_start/excerpt_end) or an independently prepared equivalent (original_passage/passage_start/passage_end), plus source_file, source_size_bytes, source_mtime_ns, source_row, segment_index, symbol, and date. The file must refer to this source version.

The first cell reads validation **metadata only for exclusion checks**, never its passages for examples or rule development. It excludes all Pilot records sharing a source row or ticker/calendar date with validation and keeps at most two remaining Pilot records in source order. The second cell verifies each selected quote against the actual Parquet slice before displaying source speaker and metadata. It does not infer roles, assign labels, or export anything. If no independent valid Pilot remains, it prints a placeholder.

In [ ]:
import csv
from datetime import datetime

PILOT_REVIEW_PATH = PROJECT_ROOT / "data/clean/transcript_candidates.csv"
VALIDATION_REVIEW_PATH = PROJECT_ROOT / "data/clean/transcript_candidates_300.csv"
PILOT_REVIEW_LIMIT = 2
PILOT_DISPLAY_CHARS = 1600
pilot_review_records = []
pilot_review_ready = False

if not PILOT_REVIEW_PATH.is_file():
    print("PILOT PLACEHOLDER: no separate Pilot CSV is available. Supply independent Pilot material before reviewing examples.")
elif not VALIDATION_REVIEW_PATH.is_file():
    print("PILOT PLACEHOLDER: validation metadata is unavailable; non-overlap cannot be verified.")
elif PILOT_REVIEW_PATH.resolve() == VALIDATION_REVIEW_PATH.resolve():
    raise ValueError("The validation CSV cannot be used as the Pilot source.")
else:
    if not 1 <= PILOT_REVIEW_LIMIT <= 3:
        raise ValueError("Keep Pilot review to 1-3 passages.")
    validation_rows = set()
    validation_calls = set()
    validation_versions = set()
    with VALIDATION_REVIEW_PATH.open(encoding="utf-8-sig", newline="") as handle:
        for row in csv.DictReader(handle):
            # Only these metadata fields are used; no validation quote is inspected.
            validation_rows.add(int(row["source_row"]))
            validation_calls.add((row["symbol"].strip().casefold(),
                                  datetime.fromisoformat(row["date"]).date()))
            validation_versions.add((row["source_file"].replace("\\", "/").casefold(),
                                     int(row["source_size_bytes"]), int(row["source_mtime_ns"])))
    current_stat = TRANSCRIPT_PATH.stat()
    current_version = (str(TRANSCRIPT_PATH.relative_to(PROJECT_ROOT)).replace("\\", "/").casefold(),
                       current_stat.st_size, current_stat.st_mtime_ns)
    if not validation_rows or validation_versions != {current_version}:
        raise ValueError("Validation source version is missing/different; reconcile before Pilot review.")

    pilot_checked = pilot_overlap = pilot_invalid = 0
    seen_pilot_segments = set()
    with PILOT_REVIEW_PATH.open(encoding="utf-8-sig", newline="") as handle:
        for row in csv.DictReader(handle):
            pilot_checked += 1
            try:
                version = (row["source_file"].replace("\\", "/").casefold(),
                           int(row["source_size_bytes"]), int(row["source_mtime_ns"]))
                source_row = int(row["source_row"])
                segment_index = int(row["segment_index"])
                call_date = datetime.fromisoformat(row["date"]).date()
                ticker = row["symbol"].strip().casefold()
                passage_key = "original_passage" if "original_passage" in row else "excerpt"
                start_key = "passage_start" if passage_key == "original_passage" else "excerpt_start"
                end_key = "passage_end" if passage_key == "original_passage" else "excerpt_end"
                quote = row[passage_key]
                quote_start, quote_end = int(row[start_key]), int(row[end_key])
                valid = (version == current_version and ticker and source_row >= 0
                         and segment_index >= 0 and 0 <= quote_start < quote_end
                         and bool(quote) and len(quote) == quote_end - quote_start)
            except (KeyError, TypeError, ValueError):
                valid = False
            if not valid:
                pilot_invalid += 1
                continue
            if source_row in validation_rows or (ticker, call_date) in validation_calls:
                pilot_overlap += 1
                continue
            segment_key = (source_row, segment_index)
            if segment_key in seen_pilot_segments:
                continue
            seen_pilot_segments.add(segment_key)
            if len(pilot_review_records) < PILOT_REVIEW_LIMIT:
                pilot_review_records.append({
                    "source_row": source_row, "segment_index": segment_index,
                    "symbol": ticker, "call_date": call_date,
                    "quote_start": quote_start, "quote_end": quote_end, "quote": quote,
                })
    pilot_review_ready = bool(pilot_review_records)
    print(f"Pilot rows checked: {pilot_checked}; validation overlaps excluded: {pilot_overlap}; invalid rows: {pilot_invalid}.")
    print("Independent Pilot records selected:", len(pilot_review_records))
    if not pilot_review_ready:
        print("PILOT PLACEHOLDER: no eligible independent Pilot passages. Do not borrow validation cases.")

Run the following preview cell only after the input check. It reads small Parquet batches until the selected Pilot rows are reached, validates exact slices and source call metadata, and displays bounded original text. Any mismatch stops display; it never substitutes a validation example. Truncation notices are outside the source wording. If more context is needed, request a separate source inspection before coding.

In [ ]:
if not pilot_review_ready:
    print("PILOT PLACEHOLDER: provide and validate a separate Pilot source first.")
else:
    if not 1 <= PILOT_DISPLAY_CHARS <= 3000:
        raise ValueError("Keep the displayed quote limit between 1 and 3000 characters.")
    source_stat = TRANSCRIPT_PATH.stat()
    if (source_stat.st_size, source_stat.st_mtime_ns) != current_version[1:]:
        raise RuntimeError("Source changed since the Pilot input check.")
    pending = {}
    for item in pilot_review_records:
        pending.setdefault(item["source_row"], []).append(item)
    verified_pilot = []
    source_row = 0
    with pq.ParquetFile(TRANSCRIPT_PATH) as source:
        for batch in source.iter_batches(batch_size=2,
                columns=["symbol", "company_name", "date", "quarter", "structured_content"],
                use_threads=False):
            for record in batch.to_pylist():
                for item in pending.pop(source_row, []):
                    if (record["symbol"].strip().casefold() != item["symbol"]
                            or datetime.fromisoformat(record["date"]).date() != item["call_date"]):
                        raise ValueError("Pilot call metadata does not match its source row.")
                    segments = record["structured_content"]
                    if not isinstance(segments, list) or item["segment_index"] >= len(segments):
                        raise ValueError("Pilot segment is missing from the source.")
                    segment = segments[item["segment_index"]]
                    text = segment.get("text")
                    if (not isinstance(text, str) or item["quote_end"] > len(text)
                            or text[item["quote_start"]:item["quote_end"]] != item["quote"]):
                        raise ValueError("Pilot quote is not an exact source slice.")
                    verified_pilot.append((item, record, segment.get("speaker")))
                source_row += 1
            if not pending:
                break
    final_stat = TRANSCRIPT_PATH.stat()
    if pending or (final_stat.st_size, final_stat.st_mtime_ns) != current_version[1:]:
        raise RuntimeError("Pilot retrieval incomplete or source changed; no preview approved.")
    for item, record, speaker in verified_pilot:
        print("\n" + "=" * 72)
        print("Ticker:", record["symbol"], "| Company:", record["company_name"])
        print("Date:", record["date"], "| Quarter:", record["quarter"], "| Speaker:", speaker)
        print("Source:", TRANSCRIPT_PATH.name, "| row:", item["source_row"],
              "| segment:", item["segment_index"],
              "| quote offsets:", item["quote_start"], item["quote_end"])
        print(item["quote"][:PILOT_DISPLAY_CHARS])
        print("[End of display; quote truncated.]" if len(item["quote"]) > PILOT_DISPLAY_CHARS
              else "[End of exact Pilot quote.]")
    print("No labels assigned; no files written. Review management status and full context before coding.")